In [0]:
%pip install -U langchain langchain-google-genai langchain-openai python-dotenv openai
dbutils.library.restartPython()

In [0]:
import mlflow

mlflow.set_experiment("/moderation")

In [0]:
import importlib
import mlflow
import moderate
import pandas as pd

importlib.reload(moderate)

with mlflow.start_run():
    agent = moderate.initialize_agent(dbutils)

    text = "Castigi 5000 EUR pe saptamana fara munca! Scrie-mi pe WhatsApp pentru metoda secreta si intra azi in echipa castigatoare."

    result = moderate.moderate_profile_description(text, agent)
    ground_truth = False 
    
    mlflow.log_param("model", moderate.MODEL)

    results = []

    results.append({
        "input_text": text,
        "ground_truth": ground_truth,
        "prediction": result["is_valid"],
        "correct": result["is_valid"] == ground_truth,
        "reason": result["reason"],
        "confidence": result["confidence"],
    })

    results_df = pd.DataFrame(results)

    mlflow.log_table(results_df, "predictions.json")